# Segment-Wise Modeling Pipeline

This notebook evaluates linear, tree-based, and SVR models across four tea market segments using chronological 5-fold cross-validation.

Segments:
- High Grown
- Low Grown
- Off-Grade
- Dust

Metrics per model x segment:
- RMSE
- MAE
- MAPE
- R2

In [1]:
import warnings
warnings.filterwarnings('ignore')

import importlib
import sys
import numpy as np
import pandas as pd
from pathlib import Path

PROCESSING_DIR = Path.cwd().resolve()
if PROCESSING_DIR.name != 'processing':
    alt_dir = PROCESSING_DIR / 'src' / 'processing'
    if alt_dir.exists():
        PROCESSING_DIR = alt_dir
if str(PROCESSING_DIR) not in sys.path:
    sys.path.insert(0, str(PROCESSING_DIR))

import modeling_common
importlib.reload(modeling_common)

from modeling_common import (
    SEED,
    TARGET,
    build_model_registry,
    get_segment_data,
    load_preprocessed_df,
    resolve_project_root,
    run_timeseries_cv,
)

np.random.seed(SEED)
print('Shared module loaded from:', PROCESSING_DIR / 'modeling_common.py')

Shared module loaded from: C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\src\processing\modeling_common.py


In [2]:
ROOT = resolve_project_root()
df, DATA_PATH = load_preprocessed_df(ROOT, keep_target_only=True)

print('Loaded:', DATA_PATH)
print('Data shape:', df.shape)
display(df.head(3))

Loaded: C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\data\processed\tea_preprocessed_v2.csv
Data shape: (2886, 244)


,sale_id,sale_number,sale_year,sale_month,sale_date_raw,table_source,elevation,category_type,grade,tier,...,poly2__western_high__precipitation_sum_total^2,poly2__western_high__precipitation_sum_total__x__nuwara_eliya__precipitation_sum_total,poly2__western_high__precipitation_sum_total__x__uva_udapussellawa__precipitation_sum_total,poly2__western_high__precipitation_sum_total__x__low_grown__precipitation_sum_total,poly2__nuwara_eliya__precipitation_sum_total^2,poly2__nuwara_eliya__precipitation_sum_total__x__uva_udapussellawa__precipitation_sum_total,poly2__nuwara_eliya__precipitation_sum_total__x__low_grown__precipitation_sum_total,poly2__uva_udapussellawa__precipitation_sum_total^2,poly2__uva_udapussellawa__precipitation_sum_total__x__low_grown__precipitation_sum_total,poly2__low_grown__precipitation_sum_total^2
0,SALE_01_2026,1,2026.0,January,06TH/07TH January 2026,06_offgrade_dust,high_grown,dust,NaN,NaN,...,151.29,243.54,316.11,202.95,392.04,508.86,326.7,660.49,424.05,272.25
1,SALE_01_2026,1,2026.0,January,06TH/07TH January 2026,06_offgrade_dust,medium_grown,dust,NaN,NaN,...,151.29,243.54,316.11,202.95,392.04,508.86,326.7,660.49,424.05,272.25
2,SALE_01_2026,1,2026.0,January,06TH/07TH January 2026,06_offgrade_dust,low_grown,dust,NaN,NaN,...,151.29,243.54,316.11,202.95,392.04,508.86,326.7,660.49,424.05,272.25


In [3]:
segment_data = get_segment_data(df, target=TARGET)

for seg, (sdf, feature_cols) in segment_data.items():
    print(f"{seg:<10} rows={len(sdf):4d} features={len(feature_cols):3d}")

High Grown rows= 516 features=220
Low Grown  rows=1348 features=220
Off-Grade  rows= 499 features=220
Dust       rows= 523 features=220


In [4]:
# Model registry comes from src/processing/modeling_common.py.
models = build_model_registry(seed=SEED)
print('Registered models:', list(models.keys()))

Registered models: ['Ridge', 'Lasso', 'ElasticNet', 'Random Forest', 'Gradient Boosting', 'SVR (RBF)', 'XGBoost', 'LightGBM']


In [5]:
# Cross-validation utilities are imported from src/processing/modeling_common.py.
print('run_timeseries_cv is available from shared module.')

run_timeseries_cv is available from shared module.


In [6]:
models = build_model_registry(seed=SEED)
all_folds = []

for seg_name, (sdf, feature_cols) in segment_data.items():
    print(f'\nRunning segment: {seg_name} | rows={len(sdf)} | features={len(feature_cols)}')

    # Need enough rows for 5 folds in TimeSeriesSplit.
    if len(sdf) < 20:
        print(f'  Skipped {seg_name}: not enough rows for stable 5-fold CV.')
        continue

    for model_name, model_obj in models.items():
        try:
            fold_df = run_timeseries_cv(sdf, feature_cols, model_name, model_obj, target=TARGET, k=5)
            fold_df.insert(0, 'Segment', seg_name)
            all_folds.append(fold_df)
            print(f'  Done: {model_name}')
        except Exception as ex:
            print(f'  Failed: {model_name} -> {ex}')

results_folds = pd.concat(all_folds, ignore_index=True) if all_folds else pd.DataFrame()
print('\nFold-level result shape:', results_folds.shape)
display(results_folds.head(10))


Running segment: High Grown | rows=516 | features=220
  Done: Ridge
  Done: Lasso
  Done: ElasticNet
  Done: Random Forest
  Done: Gradient Boosting
  Done: SVR (RBF)
  Done: XGBoost
  Done: LightGBM

Running segment: Low Grown | rows=1348 | features=220
  Done: Ridge
  Done: Lasso
  Done: ElasticNet
  Done: Random Forest
  Done: Gradient Boosting
  Done: SVR (RBF)
  Done: XGBoost
  Done: LightGBM

Running segment: Off-Grade | rows=499 | features=220
  Done: Ridge
  Done: Lasso
  Done: ElasticNet
  Done: Random Forest
  Done: Gradient Boosting
  Done: SVR (RBF)
  Done: XGBoost
  Done: LightGBM

Running segment: Dust | rows=523 | features=220
  Done: Ridge
  Done: Lasso
  Done: ElasticNet
  Done: Random Forest
  Done: Gradient Boosting
  Done: SVR (RBF)
  Done: XGBoost
  Done: LightGBM

Fold-level result shape: (160, 9)


,Segment,Model,Fold,RMSE,MAE,MAPE,R2,n_train,n_test
0,High Grown,Ridge,1,340.328907,277.262755,28.028924,-0.295326,86,86
1,High Grown,Ridge,2,383.170404,315.105493,26.968812,-1.414767,172,86
2,High Grown,Ridge,3,300.683618,251.323414,28.696845,-0.080755,258,86
3,High Grown,Ridge,4,351.888751,291.445513,30.484867,-0.786680,344,86
4,High Grown,Ridge,5,356.638684,311.436870,33.438633,-0.178310,430,86
5,High Grown,Lasso,1,442.010088,353.055976,41.704871,-0.989357,86,86
6,High Grown,Lasso,2,445.558183,398.457402,32.735702,-2.971024,172,86
7,High Grown,Lasso,3,297.178734,229.881078,28.646858,-0.120276,258,86
8,High Grown,Lasso,4,267.664703,218.637165,22.702301,-0.162604,344,86
9,High Grown,Lasso,5,319.790478,259.702710,31.273154,-0.049823,430,86


In [7]:
if results_folds.empty:
    print('No CV results generated.')
else:
    summary = (
        results_folds
        .groupby(['Segment', 'Model'], as_index=False)
        .agg(
            RMSE=('RMSE', 'mean'),
            MAE=('MAE', 'mean'),
            MAPE=('MAPE', 'mean'),
            R2=('R2', 'mean')
        )
        .sort_values(['Segment', 'RMSE'])
        .reset_index(drop=True)
    )

    print('Unified CV summary (mean over 5 folds):')
    display(summary)

    out_dir = ROOT / 'data' / 'processed'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / 'unified_model_cv_results.csv'
    summary.to_csv(out_path, index=False)
    print(f'Saved summary to: {out_path}')

Unified CV summary (mean over 5 folds):


,Segment,Model,RMSE,MAE,MAPE,R2
0,Dust,Random Forest,204.115181,161.450717,18.227987,-0.032219
1,Dust,LightGBM,215.498838,170.299247,19.489139,-0.130380
2,Dust,XGBoost,229.388503,183.323919,20.778404,-0.386907
3,Dust,Gradient Boosting,247.619155,200.765646,23.283546,-0.505438
4,Dust,SVR (RBF),253.429454,208.819509,20.971330,-0.594788
5,Dust,Lasso,276.204135,223.959499,24.495222,-1.037253
6,Dust,Ridge,311.024547,255.848373,27.297273,-1.318991
7,Dust,ElasticNet,337.634678,269.184049,29.277613,-1.827045
8,High Grown,SVR (RBF),270.214103,206.746714,24.270505,-0.038817
9,High Grown,Random Forest,273.424223,198.204100,24.063034,-0.068567


Saved summary to: C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\data\processed\unified_model_cv_results.csv
